# Unlimited-OCR (Baidu, 2026) + LoRA for **Sinhala Handwritten** OCR

**Literature-review baseline track.** Is a recent general-purpose ("multilingual") document
OCR model competitive on Sinhala handwriting, zero-shot and after fine-tuning? Guide:
`GuideForUnlimitedOCR/` (read `rules.md` before editing).

| | |
|---|---|
| Model | `baidu/Unlimited-OCR` (arXiv:2606.23050), 3B, MIT. DeepSeek-OCR encoder (SAM-B + CLIP-L) + DeepSeek-V2 **MoE** decoder whose attention is replaced by *Reference Sliding Window Attention* (window 128) for very long outputs |
| Data | `Datasets/handwritten-data` = **SinOCR-Handwritten** (train 908 / test 227) - identity asserted against IJDAR Table 2 |
| Metric | CER = (S+D+I)/N on Unicode code points (IJDAR Eq. 1), corpus-level primary - the same definition behind TrOCR 0.5253, DeepSeek 0.7059 and LightOnOCR 0.2636 |
| Method | QLoRA on the **decoder** (the stock remote code runs the vision tower under `torch.no_grad()`, see rules 7-8); vision LoRA is an opt-in ablation |

## Prerequisites (Kaggle)
1. **Accelerator: `GPU T4 x2`** (never P100, never TPU). 2. **Internet: ON**.
3. **Add Input -> `handwritten-data2`** (SinOCR-Handwritten; found by structure, any name).
4. *Optional:* `SinFundOCRClean` (from `flReadyPipeline/scripts/build_clean_ocr_data.py`) for an
   in-domain test on SinFUND form crops (Cell 12).
5. **Fresh session.** This needs `transformers==4.57.1`; LightOnOCR needs 5.0.0 and DeepSeek 4.56.2.

## Read the output in this order
Cell 1 revision + versions -> Cell 2 `MATCH` x2 -> Cell 4 config asserts -> Cell 7 smoke test ->
Cell 9 zero-shot (is the raw output *Sinhala script*?) -> Cell 11 scoreboard.

In [ ]:
# ============================================================================
# Cell 1 — Configuration, environment, and the fp16/bf16 decision
# ============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # single GPU on purpose (DeepSeek rule 2)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

IN_KAGGLE = os.path.isdir("/kaggle")
# Verification-only switch used to execute this notebook on a laptop CPU before a Kaggle
# session is spent (rule 16). Never set on Kaggle.
LOCAL_CPU_CHECK = (not IN_KAGGLE) and os.environ.get("UOCR_LOCAL_CPU_CHECK") == "1"

if IN_KAGGLE:
    # transformers 4.57.1 is what the model card tests. Its remote code imports
    # `transformers.utils.import_utils.is_torch_fx_available`, which v5 REMOVED, so a
    # floating `-U` would make the model unloadable. Pinned, never -U (rule 1).
    get_ipython().system('pip install -q "transformers==4.57.1" "peft==0.17.1" "accelerate>=0.34.0" "bitsandbytes>=0.46.1"')
    get_ipython().system('pip install -q einops addict easydict jiwer')
    # PEFT's is_torchao_available() RAISES on Kaggle's torchao 0.10.0 (DeepSeek rule 6a).
    get_ipython().system('pip install -q --no-deps --upgrade "torchao>=0.16.0"')

import sys, random, json, math, time, gc, csv, copy, functools, contextlib
print = functools.partial(print, flush=True)       # a killed kernel keeps its log
import numpy as np
import torch
import transformers, peft

NOTEBOOK_VERSION = "2026-10-20.1"
print("=" * 70)
print(f"unlimitedOcrForHandWrittenText.ipynb   revision {NOTEBOOK_VERSION}")
print("=" * 70)
print(f"transformers {transformers.__version__} | peft {peft.__version__} | torch {torch.__version__}")
if not transformers.__version__.startswith("4.57"):
    raise RuntimeError(f"need transformers==4.57.1, got {transformers.__version__}. Run Cell 1 once, "
                       f"then Kaggle -> Run -> Restart session, then Run All.")

# ---------------------------------------------------------------- CONFIG ----
CFG = dict(
    # --- what to run --------------------------------------------------------
    smoke_test            = False,  # True => tiny end-to-end pass; numbers meaningless
    run_zero_shot         = True,   # prompt selection on VAL, then the zero-shot TEST row
    do_train              = True,
    run_final_test_eval   = True,
    run_sinfund_eval      = True,   # Cell 12, only if SinFundOCRClean is attached
    sinfund_eval_n        = 500,    # seeded sample of its 1,418 test crops (0 = all)

    # --- model --------------------------------------------------------------
    base_repo             = "baidu/Unlimited-OCR",
    # Pinned: trust_remote_code would otherwise run whatever the repo serves today.
    base_revision         = "8249d20b2822a7d8a4c22c8d8c9069b5ac36bda3",
    load_in_4bit          = True,   # QLoRA, as the DeepSeek track (same 3B-MoE family)
    force_dtype           = "auto", # "auto": fp16 on T4 (sm_75), bf16 on Ampere+ | "fp16" | "bf16" | "fp32"
    # The DeepSeek-OCR Sinhala recipe (arXiv:2606.29378 Exp 1: r=16, alpha=16) adapted only
    # the decoder and reached CER 0.0302 on printed Sinhala - decoder-only LoRA is viable.
    lora_r                = 16,
    lora_alpha            = 16,
    lora_dropout          = 0.05,   # the DeepSeek track overfit at 817 samples with 0
    lora_targets          = "attn+mlp",  # "attn+mlp" (q/k/v/o + gate/up/down incl. experts) | "attn"
    # ABLATION. The stock forward() runs SAM/CLIP/projector under torch.no_grad(), so vision
    # LoRA would get ZERO gradient. True patches that no_grad out (Cell 5) and the smoke test
    # asserts the vision LoRA really receives gradients. Expect much more VRAM.
    adapt_vision_tower    = False,

    # --- input representation: the model's OWN infer() defaults ("gundam") ---
    base_size             = 1024,
    image_size            = 640,
    crop_mode             = True,
    max_crops             = 32,     # infer() calls dynamic_preprocess(image) -> max_num=32
                                    # (DeepSeek's collator capped at 9 -> up to 2.42x stretch)
    # Zero-shot candidates, chosen on VAL (Cell 9): the official document prompt and the
    # DeepSeek-OCR plain-OCR prompt. "" => fine-tune with the val-selected one.
    prompts               = ("<image>document parsing.", "<image>\nFree OCR."),
    prompt                = "",
    # Official README inference settings (also in generation_config.json).
    no_repeat_ngram_size  = 35,
    ngram_window          = 128,

    # --- data -----------------------------------------------------------------
    val_fraction          = 0.10,   # carved out of TRAIN; test is touched only in Cell 11
    augment               = True,

    # --- optimisation: the DeepSeek track's recipe (their Exp 1 values) -------
    epochs                = 3,
    per_device_batch      = 1,      # up to 32 tiles per crop -> long sequences; accum keeps
    grad_accum            = 8,      #   the effective batch at 8 (DeepSeek: 2 x 4)
    lr                    = 2e-4,
    lr_scheduler          = "linear",
    warmup_steps          = 10,
    weight_decay          = 0.001,
    max_grad_norm         = 1.0,
    optim                 = "adamw_8bit",
    grad_checkpointing    = True,

    # --- validation / early stopping ------------------------------------------
    eval_every_steps      = 25,     # optimizer steps (= 200 samples at batch 1 x accum 8)
    val_eval_n            = 64,
    early_stop_patience   = 4,
    prompt_select_n       = 48,     # VAL crops per candidate prompt

    # --- budgets ---------------------------------------------------------------
    train_hours_budget    = 6.0,
    eval_max_new_tokens   = 0,      # 0 => 2 x longest measured target + 16
    zeroshot_n            = 227,    # all test rows
    # Also report the NON-selected prompt(s) on TEST as secondary rows (never used to select
    # anything). Needed because corpus CER scores an all-empty output at exactly 1.0, so an
    # empty-output prompt can win selection against one that reads partially (learnings U1).
    zeroshot_all_prompts  = True,
    zeroshot_max_new_tok  = 192,    # un-finetuned models ramble; bound the runtime
    seed                  = 3407,

    assert_dataset_identity = True,  # IJDAR Table 2: 908/227 images, 17122/4029 code points
    push_to_hub           = False,   # Cell 13; needs a Kaggle secret HF_TOKEN (write)
    hub_repo_name         = "Model_UnlimitedOCR-Sin-Handwritten-Text",
)
if LOCAL_CPU_CHECK:
    # A laptop CPU can only verify the pipeline end to end, never train it: force the smoke
    # path and short generations (rule 16). Nothing produced in this mode is a result.
    CFG.update(smoke_test=True, zeroshot_max_new_tok=48, eval_max_new_tokens=48)
if CFG["smoke_test"]:
    CFG.update(epochs=1, zeroshot_n=6, val_eval_n=4, prompt_select_n=4, eval_every_steps=2,
               grad_accum=1, train_hours_budget=0.4, sinfund_eval_n=6)
    print(">>> SMOKE TEST MODE: tiny run, results are NOT meaningful\n")
if LOCAL_CPU_CHECK:
    # ~3 min per generated crop on a laptop CPU: exercise every code path on as few crops as possible.
    CFG.update(prompt_select_n=2, zeroshot_n=2, val_eval_n=2, sinfund_eval_n=4)

OUTPUT_DIR = "/kaggle/working" if IN_KAGGLE else os.environ.get("UOCR_OUTPUT_DIR", ".")
RUN_DIR    = os.path.join(OUTPUT_DIR, "unlimitedocr_hw_run")
BEST_DIR   = os.path.join(RUN_DIR, "best_adapter")
os.makedirs(RUN_DIR, exist_ok=True)

# ------------------------------------------------------- GPU: fail fast ----
if LOCAL_CPU_CHECK:
    # The remote code calls .cuda() unconditionally; for a CPU verification run, make that a
    # no-op. bf16 halves RAM (6.7 GB of weights); 4-bit and 8-bit Adam need CUDA.
    torch.Tensor.cuda = lambda self, *a, **k: self
    DEVICE, COMPUTE_DTYPE, USE_FP16_AMP = "cpu", torch.bfloat16, False
    CFG.update(load_in_4bit=False, optim="adamw_torch")
    print(">>> LOCAL CPU CHECK: verification only - no 4-bit, no fp16; numbers are NOT reported\n")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU. Kaggle -> Settings -> Accelerator -> 'GPU T4 x2'. Not TPU.")
    cap = torch.cuda.get_device_capability(0)
    ARCH_LIST, DEV_ARCH = torch.cuda.get_arch_list(), f"sm_{cap[0]}{cap[1]}"
    print(f"GPU {torch.cuda.get_device_name(0)} (CC {cap[0]}.{cap[1]}), "
          f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB | torch archs {ARCH_LIST}")
    if DEV_ARCH not in ARCH_LIST:
        raise RuntimeError(f"This PyTorch has no kernels for {DEV_ARCH}. Use 'GPU T4 x2' (sm_75), "
                           f"never P100 (sm_60): it died mid-load in the DeepSeek track.")
    HAS_NATIVE_BF16 = cap[0] >= 8
    # is_bf16_supported() reports True on T4 via EMULATION (DeepSeek rule 7) - decide by
    # compute capability. The same model family trained cleanly in fp16 on T4 (CER 0.7059).
    _DT = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}
    COMPUTE_DTYPE = (_DT[CFG["force_dtype"]] if CFG["force_dtype"] in _DT
                     else (torch.bfloat16 if HAS_NATIVE_BF16 else torch.float16))
    USE_FP16_AMP = COMPUTE_DTYPE == torch.float16
    DEVICE = "cuda"
    try:
        _p = torch.randn(64, 64, device="cuda", dtype=COMPUTE_DTYPE)
        _ = (_p @ _p).sum().item(); del _p; torch.cuda.synchronize()
    except Exception as e:
        raise RuntimeError(f"a trivial {COMPUTE_DTYPE} matmul failed: {e}")
print(f"device {DEVICE} | COMPUTE_DTYPE {COMPUTE_DTYPE} | fp16 AMP {USE_FP16_AMP}")

# ---- torchao must be >= 0.16 or absent, or get_peft_model() raises (DeepSeek rule 6a) ----
import importlib, importlib.metadata, subprocess
from packaging.version import parse as _ver
def _torchao_version():
    importlib.invalidate_caches()
    try:
        return _ver(importlib.metadata.version("torchao"))
    except importlib.metadata.PackageNotFoundError:
        return None
_ta = _torchao_version()
if _ta is not None and _ta < _ver("0.16.0") and IN_KAGGLE:
    print(f"torchao {_ta} < 0.16 -> removing it (PEFT then skips its dispatcher)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
print(f"torchao: {_torchao_version() or 'not installed'}")

SEED = CFG["seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"\nrun dir: {RUN_DIR}")
print(json.dumps({k: (v if isinstance(v, (int, float, bool, str, type(None))) else str(v))
                  for k, v in CFG.items()}, indent=1))

In [ ]:
# ============================================================================
# Cell 2 — Find the datasets by STRUCTURE, verify SinOCR-Handwritten identity
# ============================================================================
import os, csv
from PIL import Image

REQUIRED = [("train", "data.csv"), ("train", "images"), ("test", "data.csv"), ("test", "images")]
IJDAR_TABLE2 = {"train": {"n": 908, "chars": 17122}, "test": {"n": 227, "chars": 4029}}

def find_roots(roots, max_depth=6):
    found = []
    for root in roots:
        if not root or not os.path.isdir(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            if dirpath[len(root):].count(os.sep) > max_depth:
                dirnames[:] = []; continue
            if all(os.path.exists(os.path.join(dirpath, a, b)) for a, b in REQUIRED):
                found.append(os.path.realpath(dirpath)); dirnames[:] = []
    return list(dict.fromkeys(found))

def print_tree(root, max_depth=4):
    for dirpath, dirnames, filenames in os.walk(root):
        d = dirpath[len(root):].count(os.sep)
        if d > max_depth:
            dirnames[:] = []; continue
        print("  " * d + (os.path.basename(dirpath) or dirpath) + "/")

def load_split(root, split):
    samples, missing = [], []
    with open(os.path.join(root, split, "data.csv"), encoding="utf-8") as f:
        for r in csv.DictReader(f):
            p = os.path.join(root, split, "images", f"{r['file_name']}.png")
            if os.path.exists(p):
                samples.append({"id": r["file_name"], "image": p, "text": r["text"],
                                "label": r.get("label") or "answer"})
            else:
                missing.append(p)
    return samples, missing

def identity_ok(root):
    for split, exp in IJDAR_TABLE2.items():
        rows, _ = load_split(root, split)
        if len(rows) != exp["n"] or sum(len(r["text"]) for r in rows) != exp["chars"]:
            return False
    return True

_roots = ["/kaggle/input"] if IN_KAGGLE else [p for p in os.environ.get("UOCR_DATA_ROOTS", "../Datasets :Datasets ").split(":")]
cands = find_roots(_roots)
# Two datasets share this layout. SinOCR-HW is recognised by its IJDAR Table-2 identity,
# SinFundOCRClean by the manifest.json its builder writes - never by directory name (rule 4).
HW_ROOT    = next((c for c in cands if identity_ok(c)), None)
CLEAN_ROOT = next((c for c in cands if os.path.exists(os.path.join(c, "manifest.json"))), None)
print(f"candidate roots: {cands}")
if HW_ROOT is None:
    if CFG["assert_dataset_identity"]:
        if IN_KAGGLE:
            print("Tree of /kaggle/input:"); print_tree("/kaggle/input")
        raise FileNotFoundError("No dataset matching SinOCR-Handwritten (IJDAR Table 2: 908/227 images, "
                                "17122/4029 code points). Attach `handwritten-data2`.")
    HW_ROOT = next(c for c in cands if c != CLEAN_ROOT)
print(f"SinOCR-Handwritten : {HW_ROOT}")
print(f"SinFundOCRClean    : {CLEAN_ROOT or 'not attached (Cell 12 will be skipped)'}")

TRAIN_ALL, m1 = load_split(HW_ROOT, "train")
TEST,      m2 = load_split(HW_ROOT, "test")
if m1 or m2:
    raise FileNotFoundError(f"missing images: {len(m1)} train, {len(m2)} test, e.g. {(m1 + m2)[:3]}")

print("\nIntegrity check vs IJDAR Table 2 (SinOCR-Handwritten):")
for name, rows in (("train", TRAIN_ALL), ("test", TEST)):
    exp = IJDAR_TABLE2[name]
    n, chars = len(rows), sum(len(r["text"]) for r in rows)        # Unicode code points
    print(f"  {name:5}: images {n:4} (paper {exp['n']})  code points {chars:5} (paper {exp['chars']})  "
          f"{'MATCH' if (n, chars) == (exp['n'], exp['chars']) else 'MISMATCH'}")

def stats(rows, label):
    q = lambda v, p: v[min(len(v) - 1, int(p * len(v)))]
    L = sorted(len(r["text"]) for r in rows)
    sizes = [Image.open(r["image"]).size for r in rows]
    W = sorted(s[0] for s in sizes); H = sorted(s[1] for s in sizes)
    A = sorted(s[0] / s[1] for s in sizes)
    print(f"\n{label} (n={len(rows)})")
    print(f"  chars/sample : min {L[0]}  med {q(L, .5)}  p95 {q(L, .95)}  max {L[-1]}")
    print(f"  width px     : min {W[0]}  med {q(W, .5)}  p95 {q(W, .95)}  max {W[-1]}")
    print(f"  height px    : min {H[0]}  med {q(H, .5)}  p95 {q(H, .95)}  max {H[-1]}")
    print(f"  aspect ratio : min {A[0]:.1f}  med {q(A, .5):.1f}  p95 {q(A, .95):.1f}  max {A[-1]:.1f}")
    tile = sum(1 for s in sizes if s[0] > 640 or s[1] > 640)
    print(f"  will TILE (a side > 640 px): {tile}/{len(rows)} ({100 * tile / len(rows):.1f}%) - "
          f"token cost measured in Cell 7")

stats(TRAIN_ALL, "TRAIN (all)")
stats(TEST, "TEST")
print("\nexample:", TRAIN_ALL[0])

In [ ]:
# ============================================================================
# Cell 3 — Train/val split (val comes out of TRAIN) + seen/unseen-text tagging
# ============================================================================
# TEST is used exactly once, in Cell 11. Prompt selection, checkpoint selection and early
# stopping all use VAL, carved out of TRAIN (DeepSeek rule 32). Same recipe and seed as the
# DeepSeek track, so the split is identical to theirs.
rng = random.Random(CFG["seed"])

def bucket(t):
    n = len(t)
    return 0 if n < 10 else 1 if n < 20 else 2 if n < 35 else 3

by_bucket = {}
for r in TRAIN_ALL:
    by_bucket.setdefault(bucket(r["text"]), []).append(r)
TRAIN, VAL = [], []
for b, rows in sorted(by_bucket.items()):
    rows = rows[:]
    rng.shuffle(rows)
    k = max(1, int(round(CFG["val_fraction"] * len(rows))))
    VAL.extend(rows[:k]); TRAIN.extend(rows[k:])
rng.shuffle(TRAIN); rng.shuffle(VAL)

assert not ({r["id"] for r in TRAIN} & {r["id"] for r in VAL}), "train/val id overlap"
assert not ({r["id"] for r in TRAIN + VAL} & {r["id"] for r in TEST}), "TRAIN/VAL LEAKED INTO TEST"
print(f"train {len(TRAIN)}   val {len(VAL)}   test {len(TEST)}  (val carved from train)")

# Disclosed, not hidden (DeepSeek rule 33): the published split is random per crop, so ~1/3 of
# test texts also occur in train. TrOCR's 0.5253 was measured under the same condition.
train_texts_all = {r["text"].strip() for r in TRAIN_ALL}
for r in TEST:
    r["seen_in_train_all"] = r["text"].strip() in train_texts_all
n_seen = sum(r["seen_in_train_all"] for r in TEST)
print(f"test rows whose exact text occurs in the 908 train rows: {n_seen}/{len(TEST)} "
      f"({100 * n_seen / len(TEST):.1f}%) - CER is reported split by seen/unseen text")

if CFG["smoke_test"]:
    TRAIN, VAL = TRAIN[:4], VAL[:4]
    print(f"\nSMOKE TEST: truncated to train {len(TRAIN)}, val {len(VAL)}")

In [ ]:
# ============================================================================
# Cell 4 — Download the pinned base model, load it (4-bit QLoRA), grab its helpers
# ============================================================================
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

BASE_DIR = os.environ.get("UOCR_MODEL_DIR") or snapshot_download(
    CFG["base_repo"], revision=CFG["base_revision"])       # public, no token needed
tokenizer = AutoTokenizer.from_pretrained(BASE_DIR, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"tokenizer: bos={tokenizer.bos_token_id} eos={tokenizer.eos_token_id} "
      f"pad={tokenizer.pad_token_id} ({tokenizer.pad_token!r})")

quant = None
if CFG["load_in_4bit"]:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)
model = AutoModel.from_pretrained(
    BASE_DIR, trust_remote_code=True, use_safetensors=True, quantization_config=quant,
    dtype=COMPUTE_DTYPE,
    device_map=({"": 0} if DEVICE == "cuda" else None),
    # REQUIRED: the decoder only defines "mha_eager" (SlidingWindowLlamaAttention); any
    # other implementation name is a KeyError inside the remote code (rule 6).
    attn_implementation="eager")
if DEVICE != "cuda":
    model = model.to(DEVICE)
print(f"loaded {CFG['base_repo']}@{CFG['base_revision'][:8]} | 4-bit {CFG['load_in_4bit']} | "
      f"{(torch.cuda.memory_allocated() / 1024**3) if DEVICE == 'cuda' else float('nan'):.2f} GB VRAM")

# ---- the architecture facts this notebook depends on: assert, do not assume ----------
_c = model.config
SLIDING_W = getattr(_c, "sliding_window_size", None) or getattr(_c, "sliding_window", None)
print(f"use_mla={getattr(_c, 'use_mla', None)}  sliding_window={SLIDING_W}  "
      f"experts={getattr(_c, 'n_routed_experts', None)} (top-{getattr(_c, 'num_experts_per_tok', None)})  "
      f"layers={_c.num_hidden_layers}")
assert getattr(_c, "use_mla", False) is False, "expected the MHA (SlidingWindowLlamaAttention) decoder"
assert SLIDING_W, "expected Reference Sliding Window Attention (sliding_window) in the config"

# CLIP's `position_ids` is a persistent buffer that is NOT in the checkpoint (the load warns
# "newly initialized"). It must be arange(num_positions), or every CLIP position lookup is wrong.
_emb = model.model.vision_model.embeddings
_want = torch.arange(_emb.num_positions, device=_emb.position_ids.device).expand((1, -1))
if torch.equal(_emb.position_ids.long(), _want):
    print("CLIP position_ids buffer: arange(%d) as required" % _emb.num_positions)
else:
    with torch.no_grad():
        _emb.position_ids.copy_(_want.to(_emb.position_ids.dtype))
    print("[fix] CLIP position_ids buffer was NOT initialised by from_pretrained -> restored to arange")

# The SAME module instance the model uses: its preprocessing is the reference we must match.
_mods = [m for m in sys.modules if m.endswith("modeling_unlimitedocr")]
if not _mods:
    raise RuntimeError("could not locate the dynamic module modeling_unlimitedocr")
UOMOD = sys.modules[_mods[0]]
BasicImageTransform  = UOMOD.BasicImageTransform
dynamic_preprocess   = UOMOD.dynamic_preprocess
text_encode          = UOMOD.text_encode
format_messages      = UOMOD.format_messages
NgramBlocker         = UOMOD.SlidingWindowNoRepeatNgramProcessor
IMAGE_TOKEN_ID       = 128815                     # hard-coded in infer()
print(f"custom module {UOMOD.__name__}: helpers imported from it")

In [ ]:
# ============================================================================
# Cell 5 — LoRA: decoder targets (verified names), optional vision ablation
# ============================================================================
import re
# PEFT's is_torchao_available() raises on old torchao (DeepSeek rule 6a): neutralise it in
# every loaded peft module, then verify.
_patched = []
for _n, _m in list(sys.modules.items()):
    if _n.startswith("peft") and _m is not None and hasattr(_m, "is_torchao_available"):
        _f = getattr(_m, "is_torchao_available")
        if hasattr(_f, "cache_clear"):
            try: _f.cache_clear()
            except Exception: pass
        setattr(_m, "is_torchao_available", lambda: False); _patched.append(_n)
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from peft.tuners.lora import torchao as _pt
_pt.is_torchao_available = lambda: False
print(f"PEFT torchao probe disabled in {len(_patched) + 1} module(s)")

# Module names verified on the real checkpoint (GuideForUnlimitedOCR/learnings.md U1):
#   decoder: model.layers.N.self_attn.{q,k,v,o}_proj (12 layers)
#            model.layers.0.mlp.{gate,up,down}_proj              (dense first layer)
#            model.layers.N.mlp.experts.M.{gate,up,down}_proj    (11 MoE layers x 64 experts)
#            model.layers.N.mlp.shared_experts.{gate,up,down}_proj
#   vision : model.sam_model.blocks.N.{attn.qkv, attn.proj, mlp.lin1, mlp.lin2}
#            model.vision_model.transformer.layers.N.{self_attn.qkv_proj, self_attn.out_proj, mlp.fc1, mlp.fc2}
#            model.projector.layers
# Full regexes (PEFT fullmatch), so no vision layer is ever caught by a decoder suffix.
R_ATTN   = r"model\.layers\.\d+\.self_attn\.(q_proj|k_proj|v_proj|o_proj)"
R_MLP    = r"model\.layers\.\d+\.mlp\.((experts\.\d+|shared_experts)\.)?(gate_proj|up_proj|down_proj)"
R_SHARED = r"model\.layers\.\d+\.mlp\.(shared_experts\.)?(gate_proj|up_proj|down_proj)"
R_VISION = (r"model\.(sam_model\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(lin1|lin2))"
            r"|vision_model\.transformer\.layers\.\d+\.(self_attn\.(qkv_proj|out_proj)|mlp\.(fc1|fc2))"
            r"|projector\.layers)")
PRESETS = {"attn+mlp": [R_ATTN, R_MLP],          # everything, incl. 2,112 routed-expert projections
           "attn+shared": [R_ATTN, R_SHARED],    # ABLATION: no routed experts -> far less capacity
           "attn": [R_ATTN]}
parts = list(PRESETS[CFG["lora_targets"]]) + ([R_VISION] if CFG["adapt_vision_tower"] else [])
TARGET_REGEX = "|".join(f"(?:{p})" for p in parts)

if CFG["adapt_vision_tower"]:
    # The stock forward() wraps SAM + CLIP + projector in `with torch.no_grad():`, so a vision
    # LoRA would silently get zero gradient. Swap the module's global `torch` for a proxy whose
    # no_grad() is a no-op; everything else passes straight through. Verified by the gradient
    # assertion in Cell 7, not assumed.
    class _TorchGradOn:
        def __getattr__(self, name): return getattr(torch, name)
        @staticmethod
        def no_grad(): return contextlib.nullcontext()
    UOMOD.torch = _TorchGradOn()
    print("vision tower: no_grad patched out (ABLATION) - expect much higher VRAM")

model = get_peft_model(model, LoraConfig(r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"],
                                         lora_dropout=CFG["lora_dropout"], bias="none",
                                         task_type="CAUSAL_LM", target_modules=TARGET_REGEX))
_lora = [n for n, _ in model.named_modules() if n.endswith("lora_A")]
_vis = [n for n in _lora if re.search(r"sam_model|vision_model|projector", n)]
print(f"LoRA modules: {len(_lora)} ({len(_vis)} vision, {len(_lora) - len(_vis)} decoder) "
      f"| preset {CFG['lora_targets']!r}")
assert _lora, "target regex matched nothing"
assert bool(_vis) == CFG["adapt_vision_tower"], "vision LoRA presence does not match the config"

if CFG["grad_checkpointing"]:
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        # NO enable_input_require_grads(): it makes the frozen embedding output a leaf that
        # requires grad, and forward() then writes the image features into it IN PLACE
        # (inputs_embeds[idx].masked_scatter_) -> "a view of a leaf Variable that requires grad
        # is being used in an in-place operation" (measured, learnings U1). Non-reentrant
        # checkpointing does not need input grads; Cell 7 asserts the LoRA still gets gradients.
        print("gradient checkpointing: ON (non-reentrant) - covers the decoder layers only")
    except Exception as e:
        print(f"[warn] gradient checkpointing unavailable ({e}); continuing with more VRAM")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")
print(f"  (DeepSeek track: 86.2M trainable on 817 samples overfit -> validation-CER early stopping)")

In [ ]:
# ============================================================================
# Cell 6 — Collator that mirrors Unlimited-OCR's own infer() + handwriting augmentation
# ============================================================================
# Adapted from the DeepSeek track's (Unsloth) collator, with these deliberate changes, each
# so that TRAINING sees exactly what infer() builds at INFERENCE (Cell 8 asserts it):
#   (1) tiling uses dynamic_preprocess(..., max_num=CFG['max_crops']) - infer() uses 32, not 9;
#   (2) the non-crop branch resizes/pads to image_size, as infer() does (not base_size);
#   (3) images are loaded with the module's own load_image (EXIF transpose) + RGB;
#   (4) the answer ends with the EOS token ID itself, not the string " <eos>" (exact, and no
#       stray space token before EOS);
#   (5) labels are built per sample before padding: prompt+image positions -> -100.
import math
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
from torch.nn.utils.rnn import pad_sequence
import random as _random

def _bg_level(img):
    g = img.convert("L"); w, h = g.size
    px = (list(g.crop((0, 0, w, 1)).getdata()) + list(g.crop((0, h - 1, w, h)).getdata())
          + list(g.crop((0, 0, 1, h)).getdata()) + list(g.crop((w - 1, 0, w, h)).getdata()))
    px.sort()
    return int(px[len(px) // 2]) if px else 255

def augment_crop(img, rnd):
    """The DeepSeek/LightOnOCR tracks' conservative augmentation: Sinhala diacritics are tiny,
    so every magnitude is mild."""
    bg = _bg_level(img); fill = (bg, bg, bg)
    if rnd.random() < 0.7:
        img = img.rotate(rnd.uniform(-1.2, 1.2), resample=Image.BILINEAR, expand=True, fillcolor=fill)
    if rnd.random() < 0.4:
        s = rnd.uniform(-0.04, 0.04); w, h = img.size
        img = img.transform((w + int(abs(s) * h), h), Image.AFFINE, (1, s, -s * h if s < 0 else 0, 0, 1, 0),
                            resample=Image.BILINEAR, fillcolor=fill)
    if rnd.random() < 0.5:
        f = rnd.uniform(0.93, 1.07)
        img = img.resize((max(8, int(img.width * f)), max(8, int(img.height * f))), Image.LANCZOS)
    if rnd.random() < 0.6: img = ImageEnhance.Brightness(img).enhance(rnd.uniform(0.90, 1.10))
    if rnd.random() < 0.6: img = ImageEnhance.Contrast(img).enhance(rnd.uniform(0.88, 1.14))
    if rnd.random() < 0.25: img = img.filter(ImageFilter.GaussianBlur(rnd.uniform(0.3, 0.7)))
    r = rnd.random()
    if r < 0.12: img = img.filter(ImageFilter.MinFilter(3))
    elif r < 0.24: img = img.filter(ImageFilter.MaxFilter(3))
    return img

class UnlimitedOCRCollator:
    PATCH, DOWNSAMPLE = 16, 4

    def __init__(self, tokenizer, dtype, image_size=640, base_size=1024, crop_mode=True,
                 max_crops=32, augment=False, seed=0):
        self.tok, self.dtype = tokenizer, dtype
        self.image_size, self.base_size, self.crop_mode, self.max_crops = image_size, base_size, crop_mode, max_crops
        self.augment, self._rnd = augment, _random.Random(seed)
        self.tf = BasicImageTransform(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), normalize=True)
        self.fill = tuple(int(x * 255) for x in self.tf.mean)
        self.bos_id = 0                                   # infer(): bos_id = 0
        self.eos_id = tokenizer.eos_token_id

    def load(self, image, augment):
        img = UOMOD.load_image(image).convert("RGB") if isinstance(image, str) else image.convert("RGB")
        return augment_crop(img, self._rnd) if augment else img

    def process_image(self, image):
        """Mirror of infer()'s image branch. Returns (ori, crops, spatial_crop, image_tokens)."""
        ori, crops = [], []
        if self.crop_mode:
            if image.size[0] <= 640 and image.size[1] <= 640:        # infer() hard-codes 640
                crop_ratio, raw = [1, 1], []
            else:
                raw, crop_ratio = dynamic_preprocess(image, min_num=2, max_num=self.max_crops,
                                                     image_size=self.image_size)
            gv = ImageOps.pad(image, (self.base_size, self.base_size), color=self.fill)
            ori.append(self.tf(gv).to(self.dtype))
            w, h = crop_ratio
            if w > 1 or h > 1:
                crops = [self.tf(c).to(self.dtype) for c in raw]
            nq = math.ceil((self.image_size // self.PATCH) / self.DOWNSAMPLE)
            nqb = math.ceil((self.base_size // self.PATCH) / self.DOWNSAMPLE)
            tok = ([IMAGE_TOKEN_ID] * nqb + [IMAGE_TOKEN_ID]) * nqb + [IMAGE_TOKEN_ID]
            if w > 1 or h > 1:
                tok += ([IMAGE_TOKEN_ID] * (nq * w) + [IMAGE_TOKEN_ID]) * (nq * h)
            spatial = [w, h]
        else:
            img = image.resize((self.image_size, self.image_size)) if self.image_size <= 640 else image
            gv = ImageOps.pad(img, (self.image_size, self.image_size), color=self.fill)
            ori.append(self.tf(gv).to(self.dtype))
            nq = math.ceil((self.image_size // self.PATCH) / self.DOWNSAMPLE)
            tok = ([IMAGE_TOKEN_ID] * nq + [IMAGE_TOKEN_ID]) * nq + [IMAGE_TOKEN_ID]
            spatial = [1, 1]
        return ori, crops, spatial, tok

    def encode(self, image, prompt, answer=None, augment=False):
        """[BOS] + prompt with <image> expanded (+ answer + EOS). Returns tensors + prompt length."""
        img = self.load(image, augment)
        ids, mask = [self.bos_id], [False]
        ori, crops, spatial = [], [], []
        pieces = prompt.split("<image>")
        assert len(pieces) == 2, "the prompt must contain exactly one <image>"
        for i, piece in enumerate(pieces):
            t = text_encode(self.tok, piece, bos=False, eos=False)
            ids += t; mask += [False] * len(t)
            if i == 0:
                o, c, s, tok = self.process_image(img)
                ori, crops, spatial = o, c, [s]
                ids += tok; mask += [True] * len(tok)
        n_prompt = len(ids)
        if answer is not None:
            t = text_encode(self.tok, answer.strip(), bos=False, eos=False) + [self.eos_id]
            ids += t; mask += [False] * len(t)
        return dict(input_ids=torch.tensor(ids, dtype=torch.long),
                    images_seq_mask=torch.tensor(mask, dtype=torch.bool),
                    images_ori=torch.stack(ori),
                    images_crop=(torch.stack(crops) if crops else
                                 torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)),
                    images_spatial_crop=torch.tensor(spatial, dtype=torch.long),
                    n_prompt=n_prompt)

    def __call__(self, rows):
        enc = [self.encode(r["image"], ACTIVE_PROMPT, r["text"], augment=self.augment) for r in rows]
        input_ids = pad_sequence([e["input_ids"] for e in enc], batch_first=True,
                                 padding_value=self.tok.pad_token_id)
        seq_mask = pad_sequence([e["images_seq_mask"] for e in enc], batch_first=True, padding_value=False)
        labels = pad_sequence([torch.cat([torch.full((e["n_prompt"],), -100), e["input_ids"][e["n_prompt"]:]])
                               for e in enc], batch_first=True, padding_value=-100)
        attn = pad_sequence([torch.ones(len(e["input_ids"]), dtype=torch.long) for e in enc],
                            batch_first=True, padding_value=0)
        return dict(input_ids=input_ids, attention_mask=attn, labels=labels,
                    images=[(e["images_crop"], e["images_ori"]) for e in enc],
                    images_seq_mask=seq_mask,
                    images_spatial_crop=torch.cat([e["images_spatial_crop"] for e in enc], dim=0))

ACTIVE_PROMPT = CFG["prompt"] or CFG["prompts"][0]     # replaced by the VAL-selected prompt in Cell 9
_cargs = (tokenizer, COMPUTE_DTYPE, CFG["image_size"], CFG["base_size"], CFG["crop_mode"], CFG["max_crops"])
train_collator = UnlimitedOCRCollator(*_cargs, augment=CFG["augment"], seed=CFG["seed"])
eval_collator  = UnlimitedOCRCollator(*_cargs, augment=False, seed=CFG["seed"])
print(f"collators ready | dtype {COMPUTE_DTYPE} | base {CFG['base_size']} image {CFG['image_size']} "
      f"crop_mode {CFG['crop_mode']} max_crops {CFG['max_crops']} | train augment {CFG['augment']}")

In [ ]:
# ============================================================================
# Cell 7 — Measured lengths/tiling, the forward() contract, worst-case smoke test
# ============================================================================
import inspect
from torch.utils.data import Dataset

class Rows(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]
train_ds = Rows(TRAIN)
q = lambda v, p: v[min(len(v) - 1, int(p * len(v)))]

# ---- target lengths in THIS tokenizer, and the 128-token ring window -----------------
tt = sorted(len(text_encode(tokenizer, r["text"], bos=False, eos=False)) for r in TRAIN + VAL)
tc = [len(text_encode(tokenizer, r["text"], bos=False, eos=False)) / max(1, len(r["text"])) for r in TRAIN + VAL]
over = sum(1 for t in tt if t + 1 > SLIDING_W)
print(f"target tokens: min {tt[0]}  med {q(tt, .5)}  p95 {q(tt, .95)}  max {tt[-1]} | "
      f"{np.mean(tc):.2f} tokens per Sinhala code point")
print(f"targets longer than the {SLIDING_W}-token generation window: {over}/{len(tt)} - beyond it a "
      f"generated token no longer sees the earliest generated tokens (the image stays visible)")
_dec = [tokenizer.decode(text_encode(tokenizer, r["text"], bos=False, eos=False)) for r in TRAIN[:200]]
_rt = sum(d == r["text"] for d, r in zip(_dec, TRAIN[:200]))
print(f"tokenizer round-trip exact on {_rt}/{min(200, len(TRAIN))} train texts (Sinhala must survive "
      f"tokenisation; the Donut track's tokenizer destroyed 86.6% of it)")
EVAL_MAX_NEW_TOKENS = CFG["eval_max_new_tokens"] or int(2 * tt[-1] + 16)
print(f"EVAL_MAX_NEW_TOKENS = {EVAL_MAX_NEW_TOKENS} "
      f"({'CFG eval_max_new_tokens' if CFG['eval_max_new_tokens'] else '2 x longest target + 16'})")

# ---- tiling cost, measured with the same function infer() uses ----------------------
_RATIOS = sorted({(i, j) for n in range(2, CFG["max_crops"] + 1) for i in range(1, n + 1)
                  for j in range(1, n + 1) if 2 <= i * j <= CFG["max_crops"]}, key=lambda x: x[0] * x[1])

def n_image_tokens(path):
    """Grid and token count exactly as dynamic_preprocess() + infer() compute them."""
    w, h = Image.open(path).size
    if w <= 640 and h <= 640:
        return 273, (1, 1)
    gw, gh = UOMOD.find_closest_aspect_ratio(w / h, _RATIOS, w, h, CFG["image_size"])
    return 273 + (10 * gw + 1) * (10 * gh), (gw, gh)
_tiles = [(r, *n_image_tokens(r["image"])) for r in TRAIN]
toks = sorted(t for _, t, _ in _tiles)
print(f"image tokens per crop: med {q(toks, .5)}  p95 {q(toks, .95)}  max {toks[-1]}  "
      f"(273 = global view only)")

# ---- forward() must accept exactly what the collator emits (DeepSeek rule 23) --------
_base = model.get_base_model() if hasattr(model, "get_base_model") else model
_need = {"input_ids", "attention_mask", "labels", "images", "images_seq_mask", "images_spatial_crop"}
_missing = _need - set(inspect.signature(_base.forward).parameters)
assert not _missing, f"forward() does not accept {_missing}: transformers or the remote code changed"
print("forward() accepts every collator key")

# ---- one WORST-CASE batch through forward + backward (DeepSeek rule 24a) --------------
_worst = [r for r, _, _ in sorted(_tiles, key=lambda x: -x[1])[:CFG["per_device_batch"]]]
batch = train_collator(_worst)
print(f"\nsmoke test on the largest crop(s): {[os.path.basename(r['image']) for r in _worst]} -> "
      f"grid {[g for r2, _, g in _tiles if r2 is _worst[0]][0]}, sequence {batch['input_ids'].shape[1]} tokens")
n_sup = int((batch["labels"] != -100).sum())
assert n_sup > 0, "every label is masked"
print(f"supervised label positions: {n_sup} (= answer tokens + EOS)")

def to_dev(b):
    out = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
    out["images"] = [(c.to(DEVICE), o.to(DEVICE)) for c, o in b["images"]]
    return out

def amp():
    if DEVICE == "cuda":
        return torch.autocast("cuda", dtype=COMPUTE_DTYPE)
    return torch.autocast("cpu", dtype=torch.bfloat16)

def answer_loss(model, b):
    """Exactly forward(labels=...).loss (asserted below), but lm_head runs only on supervised
    positions. The stock forward() materialises fp32 logits for EVERY position - image tokens
    included, up to ~3.5k x 129k vocab = 1.8 GB, ~3x that with its shift copy and log-softmax -
    which is the main OOM risk on a 15 GB T4. Used by the smoke test and by the Trainer."""
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    h = base.model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], images=b["images"],
                   images_seq_mask=b["images_seq_mask"], images_spatial_crop=b["images_spatial_crop"],
                   use_cache=False, return_dict=True)[0]
    tgt = b["labels"][:, 1:].to(h.device)
    sel = tgt != -100
    logits = base.lm_head(h[:, :-1][sel]).float()
    return torch.nn.functional.cross_entropy(logits, tgt[sel])

# ---- the lean loss must equal the stock loss (eval mode: no dropout) on the smallest crop --
_small = train_collator.__class__(*_cargs, augment=False)([min(_tiles, key=lambda x: x[1])[0]])
model.eval()
with torch.no_grad(), amp():
    _l_stock = model(**to_dev(_small)).loss.float().item()
    _l_lean = answer_loss(model, to_dev(_small)).item()
print(f"loss parity: stock forward {_l_stock:.5f} | lean {_l_lean:.5f}")
assert abs(_l_stock - _l_lean) <= 1e-2 * max(1.0, abs(_l_stock)), "lean loss != stock loss"
del _small; gc.collect()

model.train()
try:
    with amp():
        loss = answer_loss(model, to_dev(batch))
except torch.cuda.OutOfMemoryError as e:
    raise RuntimeError("OOM on the worst-case crop. In order: (1) keep adapt_vision_tower=False; "
                       "(2) CFG['max_crops']=16 (a DEVIATION from infer() - disclose it); "
                       "(3) lora_targets='attn+shared'.") from e
print(f"loss = {loss.item():.4f}")
assert torch.isfinite(loss), ("loss is NaN/Inf on the first batch. On T4 set CFG['force_dtype']='bf16' "
                              "(emulated, slower, numerically safe) and re-run.")
loss.backward()
_g = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is not None and p.grad.abs().sum() > 0]
_gv = [n for n in _g if re.search(r"sam_model|vision_model|projector", n)]
print(f"parameters with non-zero gradient: {len(_g)} ({len(_gv)} in the vision tower)")
assert _g, "no gradient reached the LoRA parameters"
if CFG["adapt_vision_tower"]:
    assert _gv, "vision LoRA got NO gradient: the no_grad patch did not take effect"
else:
    assert not _gv, "unexpected vision gradients"
model.zero_grad(set_to_none=True)
if DEVICE == "cuda":
    print(f"peak VRAM after smoke test: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB of ~14.6")
del loss, batch; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()
print("smoke test PASSED")

In [ ]:
# ============================================================================
# Cell 8 — Inference (exact infer() parity), post-processing, metrics
# ============================================================================
import re, unicodedata, collections
from transformers import LogitsProcessorList
STOP_STR = "<｜end▁of▁sentence｜>"

@torch.no_grad()
def ocr_predict(row, prompt=None, max_new_tokens=None, use_adapter=True):
    """Custom generate() call (DeepSeek rule 14) that reproduces infer(eval_mode=True):
    same inputs (Cell 6 collator, asserted below), greedy, the README's n-gram blocker, and
    the Reference-Sliding-Window switch infer() makes before generating: the decoder keeps the
    whole prefill (image + prompt) and a ring buffer of the last `sliding_window` generated
    tokens. Setting config.sliding_window=None stops DynamicCache truncating the prefill."""
    s = eval_collator.encode(row["image"], prompt or ACTIVE_PROMPT, answer=None, augment=False)
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    cfg = base.config
    saved = getattr(cfg, "sliding_window", None)
    cfg._ring_window, cfg.sliding_window = SLIDING_W, None
    kw = dict(images=[(s["images_crop"].to(DEVICE), s["images_ori"].to(DEVICE))],
              images_seq_mask=s["images_seq_mask"].unsqueeze(0).to(DEVICE),
              images_spatial_crop=s["images_spatial_crop"].unsqueeze(0) if s["images_spatial_crop"].dim() == 1
                                  else s["images_spatial_crop"],
              do_sample=False, num_beams=1,            # greedy; never temperature=0.0 (DeepSeek rule 15)
              eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
              max_new_tokens=max_new_tokens or EVAL_MAX_NEW_TOKENS, use_cache=True)
    if CFG["no_repeat_ngram_size"] and CFG["ngram_window"]:
        kw["logits_processor"] = LogitsProcessorList([NgramBlocker(CFG["no_repeat_ngram_size"], CFG["ngram_window"])])
    input_ids = s["input_ids"].unsqueeze(0).to(DEVICE)
    model.eval()
    ctx = model.disable_adapter() if (not use_adapter and hasattr(model, "disable_adapter")) else contextlib.nullcontext()
    try:
        with ctx, amp():
            try:
                out = model.generate(input_ids, **kw)
            except TypeError as e:
                # PEFT's wrapper rejected the custom kwargs: the base module still has the LoRA
                # layers swapped in, so this is equivalent, not a downgrade (DeepSeek rule 17).
                if not getattr(ocr_predict, "_warned", False):
                    print(f"[warn] wrapper generate() rejected kwargs ({e}); using the base module")
                    ocr_predict._warned = True
                out = base.generate(input_ids, **kw)
    finally:
        cfg.sliding_window = saved
    # Decoded exactly as infer(eval_mode=True): special tokens KEPT. <|det|>/<|/det|> are special
    # tokens, so skip_special_tokens=True would drop the markers but keep their "text [bbox]"
    # payload inside the prediction; remove_det() below needs the markers to strip it.
    ocr_predict.last_n_new = int(out.shape[1] - input_ids.shape[1])     # 1 = EOS straight away
    raw = tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=False)
    if raw.endswith(STOP_STR):
        raw = raw[:-len(STOP_STR)]
    return raw.strip()

# ---------------------------------------------------------------- post-processing --
# Official post-processing from the Unlimited-OCR README (used for OmniDocBench): drops
# <|det|>type [bbox]<|/det|> layout markers. Then only wrappers are stripped - characters are
# never edited, and ZWJ (U+200D, Sinhala conjuncts) is never touched (DeepSeek rule 37).
IMG_BLOCK_RE = re.compile(r"<\|det\|>\s*image\b")
DET_RE = re.compile(r'<\|det\|>([^<\s]+)(?:\s*\[[^\]]*\])?\s*<\|/det\|>(.*)', re.DOTALL)
def remove_det(raw):
    blocks, cur = [], None
    for line in raw.splitlines():
        line = line.rstrip()
        if not line:
            continue
        m = DET_RE.match(line)
        if m:
            category, content = m.group(1).strip(), m.group(2).strip()
            if category == "image":
                continue
            if cur is not None:
                blocks.append(cur)
            cur = [content] if content else []
            continue
        if cur is None:
            cur = []
        cur.append(line)
    if cur is not None:
        blocks.append(cur)
    return "\n\n".join("\n".join(b) for b in blocks).strip()

def normalize_text(t):
    """Applied to BOTH reference and hypothesis: whitespace only, U+200B dropped."""
    return re.sub(r"\s+", " ", (t or "").replace("​", "")).strip()

def strip_scaffolding(t):
    t = t or ""
    for s in (STOP_STR, "<｜▁pad▁｜>", "<｜begin▁of▁sentence｜>", "</s>"):
        t = t.replace(s, "")
    t = re.sub(r"<\|[^>]*\|>", "", t)                       # leftover <|ref|>-style markers
    return re.sub(r"^\s*```[a-zA-Z]*|```\s*$", "", t)

def postprocess(raw):
    return normalize_text(strip_scaffolding(remove_det(raw or "")))

def script_profile(texts):
    """Share of predicted letters per Unicode script: the SCRIPT of garbage output was the
    fastest diagnosis in the DeepSeek and Donut tracks (DeepSeek rule 36)."""
    c = collections.Counter()
    for t in texts:
        for ch in t:
            if ch.isspace() or not ch.isprintable():
                continue
            name = unicodedata.name(ch, "")
            c[(name.split() or ["?"])[0] if name else "?"] += 1
    tot = sum(c.values()) or 1
    return {k: round(v / tot, 3) for k, v in c.most_common(5)}

# ---------------------------------------------------------------- metrics -----------
def edit_counts(ref, hyp):
    """Levenshtein with an operation breakdown -> (S, D, I, C, N); N = reference code points
    (IJDAR Eq. 1)."""
    r, h = list(ref), list(hyp)
    n, m = len(r), len(h)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): d[i][0] = i
    for j in range(m + 1): d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + (r[i - 1] != h[j - 1]))
    S = D = I = C = 0; i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and r[i - 1] == h[j - 1] and d[i][j] == d[i - 1][j - 1]:
            C += 1; i -= 1; j -= 1
        elif i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + 1:
            S += 1; i -= 1; j -= 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    return S, D, I, C, n

def word_edit_distance(ref, hyp):
    r, h = ref.split(), hyp.split()
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1): d[i][0] = i
    for j in range(len(h) + 1): d[0][j] = j
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + (r[i - 1] != h[j - 1]))
    return d[len(r)][len(h)], len(r)

def score(pairs):
    """cer_corpus = sum(S+D+I)/sum(N) (PRIMARY, IJDAR Eq. 1 - the aggregation behind the
    0.5253 / 0.7059 / 0.2636 rows); cer_macro; cer_macro_clipped (Cross-Temporal's own)."""
    per, tot_e, tot_n, tot_we, tot_wn, exact = [], 0, 0, 0, 0, 0
    for ref, hyp in pairs:
        S, D, I, C, N = edit_counts(ref, hyp)
        e = S + D + I
        per.append(e / N if N else (0.0 if not hyp else 1.0))
        tot_e += e; tot_n += N
        we, wn = word_edit_distance(ref, hyp); tot_we += we; tot_wn += wn
        exact += int(ref == hyp)
    return {"n": len(pairs), "cer_corpus": tot_e / tot_n if tot_n else float("nan"),
            "cer_macro": float(np.mean(per)) if per else float("nan"),
            "cer_macro_clipped": float(np.mean([min(c, 1.0) for c in per])) if per else float("nan"),
            "wer_corpus": tot_we / tot_wn if tot_wn else float("nan"),
            "exact_match": exact / len(pairs) if pairs else float("nan"),
            "empty_preds": sum(1 for _, h in pairs if not h.strip())}

def bootstrap_ci(pairs, n_boot=2000, seed=0):
    e, nn_ = [], []
    for ref, hyp in pairs:
        S, D, I, C, N = edit_counts(ref, hyp); e.append(S + D + I); nn_.append(N)
    e, nn_ = np.array(e, float), np.array(nn_, float)
    rs, idx = np.random.RandomState(seed), np.arange(len(e))
    vals = [e[s].sum() / max(1.0, nn_[s].sum()) for s in (rs.choice(idx, len(idx)) for _ in range(n_boot))]
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)

def evaluate(rows, label, prompt=None, max_new_tokens=None, use_adapter=True, show=3, save_as=None):
    t0, recs = time.time(), []
    for k, r in enumerate(rows):
        raw = ocr_predict(r, prompt=prompt, max_new_tokens=max_new_tokens, use_adapter=use_adapter)
        hyp = postprocess(raw)
        recs.append({"id": r["id"], "reference": normalize_text(r["text"]), "prediction": hyp, "raw": raw,
                     "n_new_tokens": ocr_predict.last_n_new,
                     "label": r.get("label"), "seen_in_train_all": r.get("seen_in_train_all")})
        if k < show:
            print(f"  [{k}] ref : {r['text']}\n      pred: {hyp}\n      raw : {raw[:160]!r}")
    pairs = [(x["reference"], x["prediction"]) for x in recs]
    m = score(pairs)
    m["cer_ci95"] = list(bootstrap_ci(pairs, seed=CFG["seed"]))
    m["scripts"] = script_profile([x["prediction"] for x in recs])
    m["seconds"] = time.time() - t0
    print(f"\n{label}: CER(corpus) {m['cer_corpus']:.4f} [95% CI {m['cer_ci95'][0]:.4f}-{m['cer_ci95'][1]:.4f}]  "
          f"macro {m['cer_macro']:.4f}  clipped {m['cer_macro_clipped']:.4f}  WER {m['wer_corpus']:.4f}  "
          f"exact {m['exact_match']:.4f}  n={m['n']}  {m['seconds'] / 60:.1f} min")
    print(f"  predicted-text scripts: {m['scripts']}")
    # Layout mode can label a whole crop as a picture: "<|det|>image [x0,y0,x1,y1]<|/det|>" and
    # nothing else. remove_det() drops image blocks (official), so the prediction is empty. That
    # is the MODEL declining to read the crop as text, not a pipeline bug (measured, learnings U1).
    m["image_block_only"] = sum(1 for x in recs if not x["prediction"] and IMG_BLOCK_RE.search(x["raw"] or ""))
    if m["image_block_only"]:
        print(f"  [model] {m['image_block_only']}/{m['n']} crops were labelled as an IMAGE region "
              f"(<|det|>image ...) and not transcribed - the model does not see this as text")
    # Raw output empty and only EOS generated: the model declined to transcribe (U1: the
    # DeepSeek-OCR prompt "Free OCR." does this on every crop). Inputs are asserted identical to
    # infer() in this cell, so this is model behaviour; empties with tokens are NOT explained.
    m["eos_only"] = sum(1 for x in recs if not x["prediction"] and not (x["raw"] or "").strip()
                        and x["n_new_tokens"] <= 1)
    if m["eos_only"]:
        print(f"  [model] {m['eos_only']}/{m['n']} crops: EOS was the first generated token - the model "
              f"declined to transcribe with this prompt")
    _other_empty = m["empty_preds"] - m["image_block_only"] - m["eos_only"]
    if _other_empty:
        print(f"  [warn] {_other_empty}/{m['n']} EMPTY predictions with no image block - a CER pinned near "
              f"1.0 with no variance is a broken pipeline, not a weak model (DeepSeek rule 35)")
    if save_as:
        with open(os.path.join(RUN_DIR, save_as), "w", encoding="utf-8") as f:
            json.dump({"label": label, "metrics": m, "records": recs}, f, ensure_ascii=False, indent=1)
    return m, recs

# ---- PARITY 1: the collator builds exactly what infer() builds -----------------------
def reference_infer_inputs(image_path, prompt):
    """Near-verbatim transcription of infer()'s input construction (modeling_unlimitedocr.py,
    crop_mode branch), kept separate from the collator so the comparison means something."""
    conversation = [{"role": "<|User|>", "content": prompt, "images": [image_path]},
                    {"role": "<|Assistant|>", "content": ""}]
    p = format_messages(conversations=conversation, sft_format="plain", system_prompt="")
    images = UOMOD.load_pil_images(conversation)
    tf = BasicImageTransform(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), normalize=True)
    base_size, image_size = 1024, 640
    toks, mask, ori, crops, spatial = [], [], [], [], []
    splits = p.split("<image>")
    for text_sep, image in zip(splits, images):
        t = text_encode(tokenizer, text_sep, bos=False, eos=False); toks += t; mask += [False] * len(t)
        if image.size[0] <= 640 and image.size[1] <= 640:
            crop_ratio = [1, 1]
        else:
            raw_crops, crop_ratio = dynamic_preprocess(image)              # infer(): defaults
        ori.append(tf(ImageOps.pad(image, (base_size, base_size),
                                   color=tuple(int(x * 255) for x in tf.mean))).to(torch.bfloat16))
        w, h = crop_ratio; spatial.append([w, h])
        if w > 1 or h > 1:
            crops += [tf(c).to(torch.bfloat16) for c in raw_crops]
        nq, nqb = math.ceil((image_size // 16) / 4), math.ceil((base_size // 16) / 4)
        tok = ([IMAGE_TOKEN_ID] * nqb + [IMAGE_TOKEN_ID]) * nqb + [IMAGE_TOKEN_ID]
        if w > 1 or h > 1:
            tok += ([IMAGE_TOKEN_ID] * (nq * w) + [IMAGE_TOKEN_ID]) * (nq * h)
        toks += tok; mask += [True] * len(tok)
    t = text_encode(tokenizer, splits[-1], bos=False, eos=False); toks += t; mask += [False] * len(t)
    return [0] + toks, [False] + mask, spatial, ori, crops

_infer_defaults = (CFG["crop_mode"], CFG["max_crops"], CFG["base_size"], CFG["image_size"]) == (True, 32, 1024, 640)
if _infer_defaults:
    _probe_rows = [TRAIN[0]] + [r for r, _, g in _tiles if g != (1, 1)][:1]
    for _r in _probe_rows:
        for _p in CFG["prompts"]:
            ids, msk, sp, ori, crops = reference_infer_inputs(_r["image"], _p)
            s = eval_collator.encode(_r["image"], _p)
            assert s["input_ids"].tolist() == ids, f"token ids differ from infer() for {_r['id']} / {_p!r}"
            assert s["images_seq_mask"].tolist() == msk, "image mask differs from infer()"
            assert s["images_spatial_crop"].tolist() == sp, "tile grid differs from infer()"
            assert torch.allclose(s["images_ori"].float(), torch.stack(ori).float(), atol=2e-2), "global view differs"
            if crops:
                assert torch.allclose(s["images_crop"].float(), torch.stack(crops).float(), atol=2e-2), "tiles differ"
    print(f"PARITY with infer(): token ids, image mask, tile grid and pixel tensors identical on "
          f"{len(_probe_rows)} crops x {len(CFG['prompts'])} prompts")
else:
    print("[note] input geometry deviates from infer() defaults -> parity check skipped; disclose it")

# ---- PARITY 2: training prompt prefix == evaluation prompt ---------------------------
_tr = train_collator.encode(TRAIN[0]["image"], ACTIVE_PROMPT, TRAIN[0]["text"], augment=False)
_ev = eval_collator.encode(TRAIN[0]["image"], ACTIVE_PROMPT)
assert _tr["input_ids"][:_tr["n_prompt"]].tolist() == _ev["input_ids"].tolist(), "train/eval prompt mismatch"
print("train prompt prefix == eval prompt: True")
try:
    import jiwer
    _S, _D, _I, _C, _N = edit_counts("හෘද සෑත්කමක්", "හෘද සෑත්")
    print(f"metric check vs jiwer: {(_S + _D + _I) / _N:.6f} vs {jiwer.cer('හෘද සෑත්කමක්', 'හෘද සෑත්'):.6f}")
except Exception as e:
    print(f"(jiwer cross-check skipped: {e})")

In [ ]:
# ============================================================================
# Cell 9 — Zero-shot: prompt chosen on VAL, then the zero-shot TEST row
# ============================================================================
# This is the literature-review question itself: does a 2026 general-purpose document OCR
# model read Sinhala handwriting WITHOUT task data? The prompt is chosen on VAL (never on
# TEST), the LoRA is disabled, and the chosen prompt is then also the one fine-tuned with.
ZS, PROMPT_SELECTION = {}, {}
if CFG["run_zero_shot"]:
    if CFG["prompt"]:
        print(f"CFG['prompt'] is fixed -> no selection: {CFG['prompt']!r}")
    else:
        sel_rows = VAL[:CFG["prompt_select_n"]]
        print(f"Prompt selection on {len(sel_rows)} VAL crops (adapter disabled, "
              f"max_new_tokens={CFG['zeroshot_max_new_tok']})\n")
        for p in CFG["prompts"]:
            print(f"--- prompt {p!r} ---")
            m, _ = evaluate(sel_rows, f"zero-shot VAL {p!r}", prompt=p, use_adapter=False,
                            max_new_tokens=CFG["zeroshot_max_new_tok"], show=2)
            PROMPT_SELECTION[p] = {k: m[k] for k in ("cer_corpus", "cer_ci95", "exact_match", "empty_preds", "scripts")}
            print()
        ACTIVE_PROMPT = min(PROMPT_SELECTION, key=lambda p: PROMPT_SELECTION[p]["cer_corpus"])  # ties -> first
        _best = PROMPT_SELECTION[ACTIVE_PROMPT]["cer_corpus"]
        if sum(abs(v["cer_corpus"] - _best) < 1e-9 for v in PROMPT_SELECTION.values()) > 1:
            print(f"TIE on VAL (CER {_best:.4f}) -> keeping the first candidate, the official prompt")
        print(f"selected on VAL: {ACTIVE_PROMPT!r} -> used for the zero-shot TEST row AND for fine-tuning")

    zs_rows = TEST[:CFG["zeroshot_n"]]
    print(f"\n--- zero-shot Unlimited-OCR on {len(zs_rows)} TEST crops ---")
    ZS["base"] = evaluate(zs_rows, "zero-shot Unlimited-OCR", use_adapter=False,
                          max_new_tokens=CFG["zeroshot_max_new_tok"], show=4,
                          save_as="zeroshot_unlimitedocr.json")[0]
    ZS["base"]["prompt"] = ACTIVE_PROMPT
    if CFG["zeroshot_all_prompts"]:
        ZS["other_prompts"] = {}
        for p in [p for p in CFG["prompts"] if p != ACTIVE_PROMPT]:
            print(f"\n--- secondary row (not selected, reported for completeness): prompt {p!r} ---")
            _safe = re.sub(r"[^a-z]+", "_", p.lower()).strip("_")
            ZS["other_prompts"][p] = evaluate(zs_rows, f"zero-shot Unlimited-OCR {p!r}", prompt=p, use_adapter=False,
                                              max_new_tokens=CFG["zeroshot_max_new_tok"], show=4,
                                              save_as=f"zeroshot_unlimitedocr_{_safe}.json")[0]
    if not ZS["base"]["scripts"] or next(iter(ZS["base"]["scripts"])) != "SINHALA":
        print("  >> the dominant predicted script is NOT Sinhala: the base model does not read this "
              "script zero-shot (compare: DeepSeek-OCR V1 zero-shot in GuideForDeepSeek)")
    with open(os.path.join(RUN_DIR, "zeroshot_summary.json"), "w", encoding="utf-8") as f:
        json.dump({"prompt_selection_on_val": PROMPT_SELECTION, "test": ZS}, f, ensure_ascii=False, indent=2)
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
else:
    print(f"run_zero_shot=False -> skipping; fine-tuning with {ACTIVE_PROMPT!r}")

In [ ]:
# ============================================================================
# Cell 10 — Train (validation-CER early stopping + wall-clock guard)
# ============================================================================
from transformers import Trainer, TrainingArguments, TrainerCallback
from safetensors.torch import load_file

class ValCERCallback(TrainerCallback):
    """Validation CER by real generation (DeepSeek rule 27: loss and CER move independently),
    keeps the best adapter, stops early. Selection happens on VAL only."""
    def __init__(self, rows, every, n, patience):
        self.rows, self.every, self.n, self.patience = rows, every, n, patience
        self.best, self.bad, self.history = float("inf"), 0, []

    def _score(self, state):
        pairs = []
        for k, r in enumerate(self.rows[:self.n]):
            hyp = postprocess(ocr_predict(r))
            pairs.append((normalize_text(r["text"]), hyp))
            if k < 2:
                print(f"    val[{k}] ref : {r['text']}\n            pred: {hyp}")
        m = score(pairs)
        self.history.append({"step": state.global_step, "cer": m["cer_corpus"],
                             "exact": m["exact_match"], "empty": m["empty_preds"]})
        print(f"  >> step {state.global_step}: val CER {m['cer_corpus']:.4f}  "
              f"exact {m['exact_match']:.3f}  empty {m['empty_preds']}/{m['n']}")
        return m["cer_corpus"]

    def on_step_end(self, args, state, control, **kw):
        if state.global_step > 0 and state.global_step % self.every == 0:
            cer = self._score(state)
            if cer < self.best - 1e-4:
                self.best, self.bad = cer, 0
                kw["model"].save_pretrained(BEST_DIR)
                print(f"     new best -> saved to {BEST_DIR}")
            else:
                self.bad += 1
                print(f"     no improvement ({self.bad}/{self.patience}); best {self.best:.4f}")
                if self.bad >= self.patience:
                    print("     early stopping"); control.should_training_stop = True
            model.train()
        return control

    def on_train_end(self, args, state, control, **kw):
        # A run that ends between two checks would otherwise never score its last weights.
        if not self.history or self.history[-1]["step"] != state.global_step:
            cer = self._score(state)
            if cer < self.best - 1e-4:
                self.best = cer; kw["model"].save_pretrained(BEST_DIR)
                print(f"     final weights are the best -> saved to {BEST_DIR}")
        return control

class WallClockGuard(TrainerCallback):
    """Kaggle sessions are capped at 12 h; stop in time for Cells 11-12."""
    def __init__(self, hours): self.deadline, self.hours = time.time() + hours * 3600, hours
    def on_step_end(self, args, state, control, **kw):
        if time.time() > self.deadline:
            print(f"\n[budget] {self.hours} h reached at step {state.global_step}; stopping.")
            control.should_training_stop = True
        return control

class AnswerLossTrainer(Trainer):
    """Same loss as forward(labels=...) (asserted in Cell 7), without full-sequence logits."""
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        loss = answer_loss(model, inputs)
        return (loss, None) if return_outputs else loss

TRAIN_INFO = {}
if CFG["do_train"]:
    print(f"fine-tuning prompt: {ACTIVE_PROMPT!r}")
    steps_per_epoch = max(1, len(TRAIN) // (CFG["per_device_batch"] * CFG["grad_accum"]))
    print(f"~{steps_per_epoch} optimizer steps/epoch x {CFG['epochs']} epochs = "
          f"~{steps_per_epoch * CFG['epochs']} steps (effective batch {CFG['per_device_batch'] * CFG['grad_accum']})")
    val_cb = ValCERCallback(VAL, CFG["eval_every_steps"], CFG["val_eval_n"], CFG["early_stop_patience"])
    args = TrainingArguments(
        output_dir                  = os.path.join(RUN_DIR, "trainer"),
        num_train_epochs            = CFG["epochs"],
        per_device_train_batch_size = CFG["per_device_batch"],
        gradient_accumulation_steps = CFG["grad_accum"],
        learning_rate               = CFG["lr"],
        warmup_steps                = CFG["warmup_steps"],
        weight_decay                = CFG["weight_decay"],
        max_grad_norm               = CFG["max_grad_norm"],
        lr_scheduler_type           = CFG["lr_scheduler"],
        logging_steps               = 1 if CFG["smoke_test"] else 5,
        eval_strategy               = "no",       # our own generation-based validation
        save_strategy               = "no",       # the callback saves the best adapter
        fp16                        = USE_FP16_AMP,
        bf16                        = COMPUTE_DTYPE == torch.bfloat16,
        use_cpu                     = LOCAL_CPU_CHECK,
        optim                       = CFG["optim"],
        remove_unused_columns       = False,      # REQUIRED: images/images_seq_mask are not inferred
        dataloader_num_workers      = 2 if IN_KAGGLE else 0,
        report_to                   = "none",
        seed                        = CFG["seed"],
        gradient_checkpointing      = False,      # already enabled in Cell 5
    )
    trainer = AnswerLossTrainer(model=model, args=args, train_dataset=train_ds, data_collator=train_collator,
                                callbacks=[val_cb, WallClockGuard(CFG["train_hours_budget"])])
    # Our loss is a per-micro-batch mean, so the Trainer must divide by grad_accum (the stock
    # DeepSeek path did the same). Explicit, not left to signature sniffing.
    trainer.model_accepts_loss_kwargs = False

    t0 = time.time()
    stats = trainer.train()
    TRAIN_INFO = {"minutes": (time.time() - t0) / 60, "steps": trainer.state.global_step,
                  "best_val_cer": val_cb.best, "prompt": ACTIVE_PROMPT}
    print(f"\ntrained in {TRAIN_INFO['minutes']:.1f} min, {TRAIN_INFO['steps']} steps | {stats.metrics}")
    if DEVICE == "cuda":
        print(f"peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
    with open(os.path.join(RUN_DIR, "train_history.json"), "w", encoding="utf-8") as f:
        json.dump({"val_history": val_cb.history, "best_val_cer": val_cb.best, "train": TRAIN_INFO,
                   "log_history": trainer.state.log_history,
                   "config": {k: str(v) for k, v in CFG.items()}}, f, ensure_ascii=False, indent=2)

    # The best checkpoint by VALIDATION CER is the model we report (never chosen on test).
    if os.path.isdir(BEST_DIR) and val_cb.best < float("inf"):
        sd = load_file(os.path.join(BEST_DIR, "adapter_model.safetensors"))
        r_ = set_peft_model_state_dict(model, sd, adapter_name="default")
        bad = [k for k in r_.unexpected_keys if "lora_" in k]
        assert not bad, f"could not restore the best adapter: {bad[:3]}"
        print(f"best adapter restored (val CER {val_cb.best:.4f})")
    else:
        print("[warn] no best checkpoint was saved; evaluating the final weights")
    # Free the optimizer state and the Trainer's references before Cells 11-12 generate (U1: the
    # laptop check swapped 12 GB and the post-training evaluation ran ~30x slower).
    trainer.optimizer = None; trainer.lr_scheduler = None
    del trainer
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
else:
    print("do_train=False -> skipping training")

In [ ]:
# ============================================================================
# Cell 11 — Final TEST evaluation (the only use of TEST) + the scoreboard
# ============================================================================
IJDAR = {"TrOCR (printed only)": 0.9940, "Tesseract (pre-trained)": 0.9493,
         "Google Vision API": 0.7532, "Tesseract (printed+handwritten)": 0.7204,
         "TrOCR (printed->handwritten)": 0.5253}
# Earlier tracks of THIS repo, same split, same corpus-CER definition (KaggleRanNotebook/).
REPO = {"DeepSeek-OCR V1 zero-shot (GuideForDeepSeek)": (5.0742, (4.3368, 5.9411)),
        "DeepSeek-OCR V1 + QLoRA (GuideForDeepSeek)": (0.7059, (0.6755, 0.7350)),
        "LightOnOCR-2-1B + QLoRA (GuideForLightOnOCR)": (0.2636, (0.2379, 0.2917))}
report, m_ft = None, None

if CFG["run_final_test_eval"] and CFG["do_train"]:
    FINAL_ROWS = TEST[:CFG["zeroshot_n"]] if CFG["smoke_test"] else TEST     # smoke: a slice, not a result
    print(f"Evaluating on {len(FINAL_ROWS)} test crops (first and only use of TEST)\n")
    m_ft, recs = evaluate(FINAL_ROWS, "FINE-TUNED Unlimited-OCR + LoRA", show=6, save_as="final_test_predictions.json")
    seen   = [(r["reference"], r["prediction"]) for r in recs if r["seen_in_train_all"]]
    unseen = [(r["reference"], r["prediction"]) for r in recs if not r["seen_in_train_all"]]
    m_seen, m_unseen = (score(seen) if seen else None), (score(unseen) if unseen else None)
    buckets = {"1-9 chars": [], "10-19": [], "20-34": [], "35+": []}
    for r in recs:
        n = len(r["reference"])
        buckets["1-9 chars" if n < 10 else "10-19" if n < 20 else "20-34" if n < 35 else "35+"].append(
            (r["reference"], r["prediction"]))

    print("\n" + "=" * 86)
    print(f"RESULTS - SinOCR-Handwritten test (n={len(FINAL_ROWS)}), CER = (S+D+I)/N on code points, corpus-level")
    print("=" * 86)
    print(f"{'Model':52} {'CER':>8}  {'95% CI':>15}  {'source':>12}")
    print("-" * 86)
    for k, v in sorted(IJDAR.items(), key=lambda x: -x[1]):
        print(f"{k:52} {v:8.4f}  {'-':>15}  {'IJDAR T5':>12}")
    _ci = lambda c: "%.4f-%.4f" % (c[0], c[1])
    for k, (v, ci) in REPO.items():
        print(f"{k:52} {v:8.4f}  {_ci(ci):>15}  {'this repo':>12}")
    if "base" in ZS:
        print(f"{'Unlimited-OCR zero-shot (val-selected prompt)':52} {ZS['base']['cer_corpus']:8.4f}  "
              f"{_ci(ZS['base']['cer_ci95']):>15}  {'this notebook':>12}")
        for _p, _z in ZS.get("other_prompts", {}).items():
            print(f"{'  zero-shot, prompt ' + repr(_p) + ' (secondary)':52} {_z['cer_corpus']:8.4f}  "
                  f"{_ci(_z['cer_ci95']):>15}  {'this notebook':>12}")
    print(f"{'Unlimited-OCR + LoRA [THIS RUN]':52} {m_ft['cer_corpus']:8.4f}  "
          f"{_ci(m_ft['cer_ci95']):>15}  {'this notebook':>12}")
    print("-" * 86)
    print("note: an EMPTY prediction scores exactly 1.0; rambling scores > 1.0 (insertions). Between two models\n"
          "      that read no Sinhala (zero-shot rows), a lower CER is NOT better reading - see `scripts` and the\n"
          "      [model] image-block / EOS-only counts printed by Cell 9.")
    if CFG["smoke_test"]:
        print(">>> SMOKE TEST: the row above is a pipeline check on a few crops, NOT a result")
    lo, hi = m_ft["cer_ci95"]
    for name, bar in (("TrOCR printed->handwritten (published SOTA)", 0.5253),
                      ("LightOnOCR-2-1B + QLoRA (this repo's best)", REPO["LightOnOCR-2-1B + QLoRA (GuideForLightOnOCR)"][0])):
        d = bar - m_ft["cer_corpus"]
        sig = "CI excludes it" if (hi < bar or lo > bar) else "CI CONTAINS it -> not a significant difference"
        print(f"vs {name} {bar:.4f}: {'BETTER' if d > 0 else 'WORSE'} by {abs(d):.4f} "
              f"({100 * abs(d) / bar:.1f}% rel.) | {sig}")

    print("\nSecondary metrics")
    print(f"  CER macro / macro-clipped : {m_ft['cer_macro']:.4f} / {m_ft['cer_macro_clipped']:.4f}")
    print(f"  WER (corpus)              : {m_ft['wer_corpus']:.4f}")
    print(f"  exact match               : {m_ft['exact_match']:.4f}")
    print(f"  empty predictions         : {m_ft['empty_preds']}/{m_ft['n']}")
    print(f"  predicted scripts         : {m_ft['scripts']}")
    if m_seen and m_unseen:
        print(f"  seen-text rows   (n={m_seen['n']:3}) : CER {m_seen['cer_corpus']:.4f}")
        print(f"  unseen-text rows (n={m_unseen['n']:3}) : CER {m_unseen['cer_corpus']:.4f}  <- new vocabulary")
    print("  by reference length:")
    for k, v in buckets.items():
        if v:
            print(f"    {k:10} n={len(v):3}  CER {score(v)['cer_corpus']:.4f}")

    def _sample_cer(r):
        S, D, I, C, N = edit_counts(r["reference"], r["prediction"])
        return (S + D + I) / max(1, N)
    print("\nWorst 8 predictions (error analysis):")
    for r in sorted(recs, key=_sample_cer, reverse=True)[:8]:
        print(f"  CER {_sample_cer(r):.2f}  ref : {r['reference']}\n            pred: {r['prediction']}")

    report = {
        "notebook": "unlimitedOcrForHandWrittenText.ipynb", "revision": NOTEBOOK_VERSION,
        "model": {"base": CFG["base_repo"], "revision": CFG["base_revision"], "prompt": ACTIVE_PROMPT,
                  "lora_targets": CFG["lora_targets"], "adapt_vision_tower": CFG["adapt_vision_tower"],
                  "load_in_4bit": CFG["load_in_4bit"], "compute_dtype": str(COMPUTE_DTYPE)},
        "dataset": {"name": "SinOCR-Handwritten", "train_used": len(TRAIN), "val": len(VAL), "test": len(TEST),
                    "identity_verified_vs_IJDAR_table2": bool(identity_ok(HW_ROOT))},
        "metric": "CER = (S+D+I)/N, N = reference Unicode code points (IJDAR Eq. 1), corpus-level",
        "results": {
            "ours_finetuned": m_ft,
            "zero_shot": ZS, "prompt_selection_on_val": PROMPT_SELECTION, "train": TRAIN_INFO,
            "seen_text": m_seen, "unseen_text": m_unseen,
            "by_length": {k: score(v)["cer_corpus"] for k, v in buckets.items() if v},
            "ijdar_table5_baselines": IJDAR, "repo_baselines": {k: v[0] for k, v in REPO.items()},
        },
        "smoke_test": CFG["smoke_test"],
        "config": {k: str(v) for k, v in CFG.items()},
    }
    with open(os.path.join(RUN_DIR, "RESULTS.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    model.save_pretrained(os.path.join(RUN_DIR, "final_adapter"))
    print(f"\nsaved: {RUN_DIR}/RESULTS.json, final_test_predictions.json, final_adapter/, best_adapter/")
elif CFG["run_final_test_eval"] and "base" in ZS:
    print("do_train=False -> the zero-shot row from Cell 9 is the result of this run:")
    print(f"  Unlimited-OCR zero-shot CER {ZS['base']['cer_corpus']:.4f}  vs TrOCR 0.5253 | LightOnOCR 0.2636")
else:
    print("run_final_test_eval=False -> skipping")

In [ ]:
# ============================================================================
# Cell 12 — OPTIONAL: in-domain test on SinFUND form crops (SinFundOCRClean test)
# ============================================================================
# SinFundOCRClean/test = every non-blank entity crop of the SinFUND TESTING forms (1,418: question,
# answer, header, other). It answers "does this OCR work on the forms the thesis is about".
#
# CONTAMINATION (GuideForFLReadyPipeline, finding P1-3/P1-13): SinOCR-Handwritten TRAIN contains
# 127 crops cut from SinFUND TEST forms (pixel audit, NCC >= 0.6), all of them ANSWER fields.
#   * zero-shot rows  : clean on every class (the base model never saw any of this);
#   * fine-tuned rows : clean on question/header/other; ANSWER is split by the audit into
#                       seen / grey / unseen, and only `unseen` is a fair number.
# Without the audit CSV every fine-tuned ANSWER number is marked contaminated, not reported.
# Placeholder answers ("<signature>", ...) are drawings: reported with and without them.
SINFUND = {}

def _find_file(name, roots):
    for root in roots:
        if root and os.path.isfile(root) and os.path.basename(root) == name:
            return root
        if root and os.path.isdir(root):
            for dp, dn, fn in os.walk(root):
                if name in fn:
                    return os.path.join(dp, name)
    return None

if CFG["run_sinfund_eval"] and CLEAN_ROOT:
    rows_all, _miss = load_split(CLEAN_ROOT, "test")
    with open(os.path.join(CLEAN_ROOT, "test", "data.csv"), encoding="utf-8") as f:
        _script = {r["file_name"]: r.get("script", "") for r in csv.DictReader(f)}
    for r in rows_all:
        r["script"] = _script.get(r["id"], "")
    rows = rows_all
    if CFG["sinfund_eval_n"] and CFG["sinfund_eval_n"] < len(rows_all):
        rows = random.Random(CFG["seed"]).sample(rows_all, CFG["sinfund_eval_n"])
    is_tag = lambda t: bool(re.fullmatch(r"\s*<[^<>]+>\s*", t or ""))

    # ---- provenance audit: which SinFUND test answers are pixel copies of SinOCR-HW train ----
    prov = _find_file("sinocr_hw_provenance.csv",
                      [os.environ.get("UOCR_PROVENANCE"), "/kaggle/input" if IN_KAGGLE else None,
                       "../flReadyPipeline/provenance", "flReadyPipeline/provenance"])
    seen_keys, grey_keys = set(), set()
    if prov:
        with open(prov, encoding="utf-8") as f:
            for a in csv.DictReader(f):
                if a["hw_split"] == "train" and a["source_entity_id"]:
                    key = f"sf_{a['source_doc']}_{int(a['source_entity_id'])}"
                    if a["source_split"] == "testing_data":    seen_keys.add(key)
                    elif a["source_split"] == "testing_data?": grey_keys.add(key)
        print(f"provenance audit: {prov} -> {len(seen_keys)} seen, {len(grey_keys)} grey-zone SinFUND test crops")
    else:
        print("provenance audit CSV not found -> fine-tuned ANSWER rows are marked CONTAMINATED and not reported "
              "(attach flReadyPipeline/provenance/sinocr_hw_provenance.csv to report the clean `unseen` subset)")
    for r in rows:
        r["prov"] = ("n/a" if r["label"] != "answer" else
                     "seen" if r["id"] in seen_keys else "grey" if r["id"] in grey_keys else
                     "unseen" if prov else "unknown")
    print(f"SinFundOCRClean test: {len(rows)} of {len(rows_all)} crops (seeded sample) | labels "
          f"{dict(collections.Counter(r['label'] for r in rows))} | answer provenance "
          f"{dict(collections.Counter(r['prov'] for r in rows if r['label'] == 'answer'))}")

    def breakdown(recs, tag, fine_tuned):
        by = {r["id"]: r for r in rows}
        out = {}
        groups = {f"{lab}": [x for x in recs if by[x["id"]]["label"] == lab] for lab in ("question", "header", "other", "answer")}
        groups["answer_text_only"] = [x for x in groups["answer"] if not is_tag(x["reference"])]
        for p in ("seen", "grey", "unseen", "unknown"):
            groups[f"answer_text_only_{p}"] = [x for x in groups["answer_text_only"] if by[x["id"]]["prov"] == p]
        groups["sinhala_script"] = [x for x in recs if by[x["id"]]["script"] == "sinhala"]
        groups["all_text_only"] = [x for x in recs if not is_tag(x["reference"])]
        groups["all"] = recs
        print(f"\n{tag}")
        for g, xs in groups.items():
            if not xs:
                continue
            pairs = [(x["reference"], x["prediction"]) for x in xs]
            m = score(pairs); m["cer_ci95"] = list(bootstrap_ci(pairs, seed=CFG["seed"]))
            dirty = fine_tuned and (g in ("answer", "answer_text_only", "all", "all_text_only", "sinhala_script")
                                    or g.endswith(("_seen", "_grey", "_unknown")))
            m["contaminated"] = bool(dirty)
            out[g] = m
            print(f"  {g:28} n={m['n']:4}  CER {m['cer_corpus']:.4f} [{m['cer_ci95'][0]:.3f}-{m['cer_ci95'][1]:.3f}]  "
                  f"exact {m['exact_match']:.3f}{'   CONTAMINATED - do not report' if dirty else ''}")
        return out

    print("\n--- zero-shot (adapter disabled): clean on every class ---")
    _, zrec = evaluate(rows, "SinFUND zero-shot Unlimited-OCR", use_adapter=False,
                       max_new_tokens=CFG["zeroshot_max_new_tok"], show=3, save_as="sinfund_zeroshot.json")
    SINFUND["zero_shot"] = breakdown(zrec, "zero-shot by group", fine_tuned=False)
    if CFG["do_train"]:
        print("\n--- fine-tuned (best adapter) ---")
        _, frec = evaluate(rows, "SinFUND fine-tuned Unlimited-OCR + LoRA", show=3, save_as="sinfund_finetuned.json")
        SINFUND["fine_tuned"] = breakdown(frec, "fine-tuned by group", fine_tuned=True)
    print("\nReference (GuideForFLReadyPipeline, LightOnOCR+QLoRA on SinFUND test ANSWER crops, no tags):")
    print("  all 0.3067 | seen-in-training 0.2143 | unseen is the honest number (see learnings P1-13)")
    SINFUND["sample"] = {"n": len(rows), "of": len(rows_all), "seed": CFG["seed"], "provenance_csv": bool(prov)}
    with open(os.path.join(RUN_DIR, "SINFUND_RESULTS.json"), "w", encoding="utf-8") as f:
        json.dump(SINFUND, f, ensure_ascii=False, indent=2)
    if report is not None:
        report["results"]["sinfund_ocr_clean"] = SINFUND
        with open(os.path.join(RUN_DIR, "RESULTS.json"), "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
    print(f"\nsaved: {RUN_DIR}/SINFUND_RESULTS.json")
else:
    print("SinFundOCRClean not attached or run_sinfund_eval=False -> skipping")

In [ ]:
# ============================================================================
# Cell 13 — OPTIONAL: push the best adapter to the Hugging Face Hub
# ============================================================================
# Kaggle: Add-ons -> Secrets -> "HF_TOKEN" (a WRITE token), toggled ON for this notebook.
if CFG["push_to_hub"] and CFG["do_train"] and not CFG["smoke_test"]:
    _tok = None
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"[kaggle_secrets unavailable: {type(e).__name__}] falling back to env HF_TOKEN")
        _tok = os.environ.get("HF_TOKEN")
    if not _tok:
        raise RuntimeError("No HF_TOKEN secret (write access). Add it, toggle it ON, re-run this cell.")
    from huggingface_hub import HfApi, login
    login(token=_tok)
    repo_id = f"{HfApi().whoami()['name']}/{CFG['hub_repo_name']}"
    if os.path.isdir(BEST_DIR):
        sd = load_file(os.path.join(BEST_DIR, "adapter_model.safetensors"))
        r_ = set_peft_model_state_dict(model, sd, adapter_name="default")
        assert not [k for k in r_.unexpected_keys if "lora_" in k], "best adapter would not load cleanly"
    _cer = f"{m_ft['cer_corpus']:.4f}" if m_ft else "see RESULTS.json"
    card = f"""---
base_model: {CFG['base_repo']}
library_name: peft
license: mit
language: [si]
tags: [ocr, sinhala, handwritten, lora, unlimited-ocr]
---

# Sinhala handwritten OCR - Unlimited-OCR + LoRA

LoRA adapter (decoder; vision tower {'adapted' if CFG['adapt_vision_tower'] else 'frozen'}) for
`{CFG['base_repo']}@{CFG['base_revision'][:8]}`, trained on SinOCR-Handwritten (train 908, of which
{len(VAL)} held out for validation). **Test CER (corpus, IJDAR Eq. 1): {_cer}** on the 227-crop test split.

Prompt: `{ACTIVE_PROMPT!r}`. Inference must use the same prompt, `crop_mode=True`, base 1024 / image 640.
"""
    model.save_pretrained(os.path.join(RUN_DIR, "hub_adapter"))
    with open(os.path.join(RUN_DIR, "hub_adapter", "README.md"), "w", encoding="utf-8") as f:
        f.write(card)
    for fn in ("RESULTS.json", "train_history.json"):
        if os.path.exists(os.path.join(RUN_DIR, fn)):
            import shutil; shutil.copy(os.path.join(RUN_DIR, fn), os.path.join(RUN_DIR, "hub_adapter", fn))
    api = HfApi()
    api.create_repo(repo_id, exist_ok=True, private=False)
    api.upload_folder(folder_path=os.path.join(RUN_DIR, "hub_adapter"), repo_id=repo_id)
    print(f"pushed -> https://huggingface.co/{repo_id}")
else:
    print("push_to_hub=False (or no training / smoke test) -> skipping")

## How to read the output, and what to bring back

**Headline number:** Cell 11 `CER(corpus)` with its 95% CI, on the same 227-crop test split
and the same CER definition as TrOCR 0.5253 (IJDAR), DeepSeek-OCR 0.7059 and LightOnOCR 0.2636.

**For the literature review, there are two answers:**
1. *Zero-shot* (Cell 9): is a 2026 "universal" document OCR model already able to read Sinhala
   handwriting? Look at the predicted-script share as well as the CER.
2. *Fine-tuned* (Cell 11): how good does it get with LoRA on 817 crops, against this repo's best?

**Sanity checks before you trust a number:** Cell 2 `MATCH` x2; Cell 7 `loss parity` and
`smoke test PASSED`; Cell 8 `PARITY with infer()`; no `EMPTY predictions` warning.

**Bring back into `KaggleRanNotebook/`** (folder `unlimitedocr_hw_run/` from the Kaggle output):
`RESULTS.json`, `train_history.json`, `final_test_predictions.json`, `zeroshot_summary.json`,
`zeroshot_unlimitedocr.json`, `SINFUND_RESULTS.json` (if Cell 12 ran), plus the executed notebook
and its console log. Then update `GuideForUnlimitedOCR/learnings.md`.

**If it underperforms, change one thing at a time (each is an ablation row):** `epochs`; `lr`
(5e-5 / 2e-4); `lora_targets="attn+shared"`; `adapt_vision_tower=True` (the vision path is frozen
by the stock code); `augment=False`; `max_crops=16` (deviation from infer(), disclose it).